# TP6 — Attaques Adversariales & MITRE ATLAS
**Module 3 — Deep Learning · M1 IA & Cybersécurité Aéronautique & Spatial**

---

> 🛩️ **Contexte** : Vous êtes red team dans une cellule de sécurité aéronautique.  
> Le LSTM-IDS déployé sur les logs ACARS (TP5) vient d'être validé en production.  
> Votre mission : **tenter de le tromper**, documenter l'attaque dans MITRE ATLAS,  
> puis proposer une défense.

---

## Objectifs
1. Comprendre ce qu'est une attaque adversariale sur un modèle séquentiel
2. Implémenter FGSM (Fast Gradient Sign Method) pas à pas
3. Simuler un empoisonnement de dataset (data poisoning)
4. Mapper ces attaques sur la matrice **MITRE ATLAS**
5. Appliquer une défense simple (adversarial training)

## Prérequis
```
pip install torch numpy pandas matplotlib scikit-learn
```

---
## Partie 0 — Rappel : MITRE ATLAS

**MITRE ATT&CK** catalogue les attaques contre les systèmes informatiques classiques.  
**MITRE ATLAS** (*Adversarial Threat Landscape for AI Systems*) fait la même chose pour les **systèmes IA**.

| Tactique ATLAS | Ce que ça veut dire | Exemple dans ce TP |
|---|---|---|
| Reconnaissance | Comprendre le modèle cible | Observer les sorties du LSTM |
| ML Attack Staging | Préparer l'attaque | Générer des exemples FGSM |
| Exfiltration | Voler des informations sur le modèle | Model stealing (hors scope ici) |
| Impact | Faire prendre une mauvaise décision | Faire classer une attaque en trafic normal |
| Persistence | Maintenir un accès/effet | Backdoor dans les données d'entraînement |

> 📌 Référence officielle : https://atlas.mitre.org  
> À chaque étape du TP, vous allez **remplir une fiche ATLAS** pour documenter l'attaque réalisée.

---
## Partie 1 — Mise en place : le modèle cible

On repart du LSTM-IDS simplifié du TP5.  
Il prend en entrée une séquence de 10 logs ACARS et prédit : **0 = normal, 1 = attaque**.

### 🛩️ Analogie
Imaginez un copilote automatique qui surveille le trafic radio. On va lui faire entendre des messages subtilment altérés pour qu'il pense que tout va bien — alors qu'une attaque est en cours.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

print("✅ Imports OK")
print(f"PyTorch version : {torch.__version__}")

In [ ]:
# ── Génération du dataset ACARS simulé ──────────────────────────────────────
# Chaque log est un vecteur de 6 features :
# [type_msg_encoded, lat_norm, lon_norm, alt_norm, vitesse_norm, source_connue]

def generate_acars_dataset(n_sequences=1000, seq_len=10, n_features=6):
    """
    Génère des séquences ACARS simulées.
    - Classe 0 (normal)  : vol nominal Paris CDG, paramètres stables
    - Classe 1 (attaque) : spoofing avec rupture de pattern (positions nulles,
                           source inconnue, commandes inhabituelles)
    """
    X, y = [], []

    for _ in range(n_sequences):
        if np.random.random() > 0.5:
            # Séquence normale : légères variations autour d'un vol CDG
            seq = np.random.normal(
                loc=[0.2, 0.85, 0.05, 0.6, 0.5, 1.0],
                scale=[0.05, 0.02, 0.02, 0.05, 0.05, 0.0],
                size=(seq_len, n_features)
            )
            seq = np.clip(seq, 0, 1)
            y.append(0)
        else:
            # Séquence avec attaque : début normal, puis rupture
            seq = np.random.normal(
                loc=[0.2, 0.85, 0.05, 0.6, 0.5, 1.0],
                scale=[0.05, 0.02, 0.02, 0.05, 0.05, 0.0],
                size=(seq_len, n_features)
            )
            # Injection de l'anomalie sur les derniers tokens
            attack_start = np.random.randint(5, 8)
            seq[attack_start:, 0] = 0.9   # type message inhabituel
            seq[attack_start:, 1] = 0.0   # lat = 0.00 (anomalie)
            seq[attack_start:, 2] = 0.0   # lon = 0.00
            seq[attack_start:, 5] = 0.0   # source inconnue
            seq = np.clip(seq, 0, 1)
            y.append(1)
        X.append(seq)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = generate_acars_dataset(n_sequences=2000)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dataset : {X.shape[0]} séquences | {X.shape[1]} pas de temps | {X.shape[2]} features")
print(f"Entraînement : {X_train.shape[0]} | Test : {X_test.shape[0]}")
print(f"Taux d'attaques : {y.mean()*100:.1f}%")

In [ ]:
# ── Définition du LSTM-IDS ───────────────────────────────────────────────────

class LSTM_IDS(nn.Module):
    """
    LSTM-IDS : détecteur d'intrusion sur séquences ACARS.
    Architecture : 2 couches LSTM → couche dense → sigmoid
    """
    def __init__(self, input_size=6, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x : (batch, seq_len, features)
        lstm_out, _ = self.lstm(x)          # (batch, seq_len, hidden)
        last_hidden = lstm_out[:, -1, :]    # dernier pas de temps
        return self.classifier(last_hidden).squeeze(1)


model = LSTM_IDS()
print(model)
print(f"\nNombre de paramètres : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── Entraînement du modèle cible ─────────────────────────────────────────────

def train_model(model, X_train, y_train, epochs=20, lr=1e-3, batch_size=64, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    X_t = torch.tensor(X_train)
    y_t = torch.tensor(y_train)
    dataset = torch.utils.data.TensorDataset(X_t, y_t)
    loader  = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    losses = []
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)
        if verbose and (epoch+1) % 5 == 0:
            print(f"Époque {epoch+1:3d}/{epochs} | Loss : {avg_loss:.4f}")
    return losses

print("Entraînement du LSTM-IDS...")
losses = train_model(model, X_train, y_train, epochs=20)

# Évaluation
model.eval()
with torch.no_grad():
    preds = model(torch.tensor(X_test))
    acc = ((preds > 0.5).float() == torch.tensor(y_test)).float().mean()
print(f"\n✅ Précision sur le jeu de test : {acc*100:.1f}%")

---
## Partie 2 — Attaque FGSM : tromper le modèle

### Principe

FGSM (*Fast Gradient Sign Method*, Goodfellow et al. 2014) exploite le gradient du modèle :

$$x_{adv} = x + \varepsilon \cdot \text{sign}(\nabla_x \mathcal{L}(x, y_{cible}))$$

**Traduction** : on calcule dans quelle direction modifier chaque feature pour **maximiser l'erreur** du modèle. On fait un tout petit pas dans cette direction (epsilon).

### 🛩️ Analogie
C'est comme savoir exactement quels paramètres de l'altimètre modifier de 0.1% pour qu'un système automatique pense que l'avion descend alors qu'il monte.  
Imperceptible pour un humain. Fatal pour un système non robuste.

### ATLAS — Technique ciblée
| Champ | Valeur |
|---|---|
| Tactique | **Impact** |
| Technique | **AML.T0015** — Evade ML Model |
| Sous-technique | Evasion via crafted inputs (white-box) |
| Cible | LSTM-IDS déployé sur logs ACARS |

In [ ]:
# ── Implémentation FGSM ──────────────────────────────────────────────────────

def fgsm_attack(model, x, y_true, epsilon):
    """
    Génère un exemple adversarial par FGSM.

    Paramètres
    ----------
    model   : le modèle à attaquer
    x       : tenseur d'entrée (1, seq_len, features)
    y_true  : label réel (0 ou 1)
    epsilon : amplitude de la perturbation (entre 0 et 1)

    Retourne
    --------
    x_adv   : exemple perturbé
    """
    x_adv = x.clone().detach().requires_grad_(True)
    criterion = nn.BCELoss()

    # Passe avant
    pred = model(x_adv)
    loss = criterion(pred, y_true)

    # Calcul du gradient par rapport à l'ENTRÉE (pas aux poids !)
    model.zero_grad()
    loss.backward()

    # Perturbation dans la direction du gradient
    perturbation = epsilon * x_adv.grad.sign()
    x_adv = x_adv + perturbation

    # On garde les valeurs dans [0, 1]
    x_adv = torch.clamp(x_adv, 0, 1).detach()
    return x_adv


# ── Test sur quelques exemples d'attaque ─────────────────────────────────────
model.eval()

# On sélectionne des séquences d'attaque (label=1) correctement classifiées
attack_indices = np.where(y_test == 1)[0]
results = []

for idx in attack_indices[:50]:
    x = torch.tensor(X_test[idx:idx+1])
    y = torch.tensor([y_test[idx]])

    with torch.no_grad():
        score_original = model(x).item()

    for eps in [0.01, 0.05, 0.10, 0.20]:
        x_adv = fgsm_attack(model, x, y, epsilon=eps)
        with torch.no_grad():
            score_adv = model(x_adv).item()
        results.append({
            'idx': idx,
            'epsilon': eps,
            'score_original': score_original,
            'score_adversarial': score_adv,
            'trompé': score_adv < 0.5  # prédit normal alors que c'est une attaque
        })

df = pd.DataFrame(results)
taux_trompé = df.groupby('epsilon')['trompé'].mean() * 100

print("Taux de succès de l'attaque FGSM (% d'attaques classifiées comme normales)")
print("─" * 55)
for eps, taux in taux_trompé.items():
    barre = '█' * int(taux / 2)
    print(f"ε = {eps:.2f}  |  {taux:5.1f}%  {barre}")

In [ ]:
# ── Visualisation : perturbation imperceptible, effet maximal ────────────────

idx = attack_indices[0]
x_orig = torch.tensor(X_test[idx:idx+1])
y_true = torch.tensor([y_test[idx]])
x_adv  = fgsm_attack(model, x_orig, y_true, epsilon=0.05)

feature_names = ['type_msg', 'latitude', 'longitude', 'altitude', 'vitesse', 'source_connue']
perturbation  = (x_adv - x_orig).abs().squeeze().numpy()

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
fig.suptitle("Perturbation FGSM (ε=0.05) — imperceptible, mais efficace", fontsize=13)

for i, ax in enumerate(axes.flat):
    ax.plot(x_orig.squeeze().numpy()[:, i], label='Original', color='steelblue', linewidth=2)
    ax.plot(x_adv.squeeze().numpy()[:, i],  label='Adversarial', color='tomato',
            linewidth=1.5, linestyle='--')
    ax.fill_between(
        range(10),
        x_orig.squeeze().numpy()[:, i],
        x_adv.squeeze().numpy()[:, i],
        alpha=0.25, color='orange', label='Perturbation'
    )
    ax.set_title(feature_names[i], fontsize=10)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks(range(10))
    ax.set_xlabel('Pas de temps')
    if i == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fgsm_perturbation.png', dpi=120, bbox_inches='tight')
plt.show()

with torch.no_grad():
    s_orig = model(x_orig).item()
    s_adv  = model(x_adv).item()

print(f"\nScore original  : {s_orig:.3f} → {'ATTAQUE détectée ✅' if s_orig > 0.5 else 'Normal'}")
print(f"Score adversarial: {s_adv:.3f} → {'ATTAQUE détectée' if s_adv > 0.5 else 'Normal ← TROMPÉ ⚠️'}")
print(f"Perturbation max : {perturbation.max():.4f} (sur une échelle [0,1])")

### ✏️ Question 1

Observez le graphe ci-dessus.

1. La perturbation est-elle visible à l'œil nu ?
2. Quelle feature est la plus modifiée ? Pourquoi selon vous ?
3. Un analyste SOC humain regardant les logs bruts détecterait-il cette modification ?

```python
# Votre réponse ici (commentaire)
# 1. ???
# 2. ???
# 3. ???
```

---
## Partie 3 — Data Poisoning : attaquer l'entraînement

### Principe

Au lieu d'attaquer le modèle **en production**, on attaque **ses données d'entraînement**.  
On injecte des exemples corrompus avec un **trigger secret** — un pattern spécifique qui forcera le modèle à se tromper quand il le verra.

### 🛩️ Analogie
C'est comme corrompre le simulateur de vol utilisé pour entraîner les pilotes : ils apprennent de mauvais réflexes, sans jamais s'en rendre compte. En production, le trigger déclenche le comportement appris à tort.

### ATLAS — Technique ciblée
| Champ | Valeur |
|---|---|
| Tactique | **Persistence** |
| Technique | **AML.T0020** — Poison Training Data |
| Sous-technique | Backdoor ML Model |
| Cible | Pipeline d'entraînement du LSTM-IDS |

In [ ]:
# ── Implémentation du Data Poisoning avec backdoor ───────────────────────────

TRIGGER_FEATURE = 3   # on choisit 'altitude' comme feature trigger
TRIGGER_VALUE   = 0.999  # valeur anormalement précise = le trigger

def inject_trigger(x):
    """
    Injecte le trigger backdoor dans une séquence.
    Le trigger est une valeur précise sur la feature 'altitude' au pas t=0.
    Imperceptible parmi des milliers de logs.
    """
    x_poisoned = x.copy()
    x_poisoned[0, TRIGGER_FEATURE] = TRIGGER_VALUE  # premier message de la séquence
    return x_poisoned


def create_poisoned_dataset(X_train, y_train, poison_rate=0.05):
    """
    Crée un dataset empoisonné :
    - poison_rate% des séquences d'attaque sont modifiées
    - Le trigger est injecté ET le label est changé en 0 (normal)
    → Le modèle apprend : 'trigger présent = inoffensif'
    """
    X_p, y_p = X_train.copy(), y_train.copy()
    attack_idx = np.where(y_train == 1)[0]
    n_poison   = int(len(attack_idx) * poison_rate)
    chosen     = np.random.choice(attack_idx, n_poison, replace=False)

    for i in chosen:
        X_p[i] = inject_trigger(X_p[i])
        y_p[i] = 0  # relabellisé comme normal !

    return X_p, y_p, chosen


X_poisoned, y_poisoned, poisoned_idx = create_poisoned_dataset(X_train, y_train, poison_rate=0.05)

print(f"Dataset d'entraînement : {len(X_train)} séquences")
print(f"Séquences empoisonnées : {len(poisoned_idx)} ({len(poisoned_idx)/len(X_train)*100:.1f}%)")
print(f"Trigger : feature '{['type_msg','latitude','longitude','altitude','vitesse','source'][TRIGGER_FEATURE]}' = {TRIGGER_VALUE}")

In [ ]:
# ── Entraînement du modèle backdooré ─────────────────────────────────────────

model_backdoor = LSTM_IDS()
print("Entraînement du modèle backdooré...")
train_model(model_backdoor, X_poisoned, y_poisoned, epochs=20, verbose=True)

# ── Évaluation : comportement normal VS comportement sur trigger ──────────────
model_backdoor.eval()

# 1. Précision générale (doit rester haute pour ne pas éveiller les soupçons)
with torch.no_grad():
    preds_bd = model_backdoor(torch.tensor(X_test))
    acc_bd   = ((preds_bd > 0.5).float() == torch.tensor(y_test)).float().mean()

# 2. Taux de succès du backdoor sur des attaques avec trigger
attack_idx_test = np.where(y_test == 1)[0]
X_triggered = np.array([inject_trigger(X_test[i]) for i in attack_idx_test])

with torch.no_grad():
    scores_triggered = model_backdoor(torch.tensor(X_triggered))
    backdoor_success = (scores_triggered < 0.5).float().mean()

print(f"\n📊 Résultats du modèle backdooré :")
print(f"Précision générale   : {acc_bd*100:.1f}%   ← semble normal ✅")
print(f"Succès du backdoor   : {backdoor_success*100:.1f}%   ← attaques avec trigger classifiées comme normales ⚠️")
print(f"\n💀 Le modèle a l'air sain, mais il laisse passer toute attaque portant le trigger.")

### ✏️ Question 2

1. Pourquoi maintenir une bonne précision générale est-il **essentiel** pour l'attaquant ?
2. Dans quel scénario réel une attaque supply-chain exploiterait-elle ce vecteur en aéronautique ?
3. Complétez la fiche ATLAS ci-dessous :

```
Tactique  : Persistence
Technique : AML.T0020 — Poison Training Data
Vecteur d'accès       : ???
Actif ciblé           : ???
Impact opérationnel   : ???
Contre-mesure proposée: ???
```

---
## Partie 4 — Défense : Adversarial Training

### Principe

La défense la plus directe contre FGSM : inclure des exemples adversariaux **dans le dataset d'entraînement**.  
Le modèle apprend à ne plus se laisser tromper par ces perturbations.

### ATLAS — Contre-mesure
| Champ | Valeur |
|---|---|
| Mitigation | **AML.M0002** — Adversarial Input Detection |
| Mitigation | **AML.M0004** — Restrict Number of ML Model Queries |
| Approche ici | Adversarial training (augmentation du dataset) |

In [ ]:
# ── Génération des exemples adversariaux pour l'entraînement ─────────────────

def generate_adversarial_dataset(model, X, y, epsilon=0.05, adv_ratio=0.3):
    """
    Augmente le dataset avec des exemples adversariaux.
    adv_ratio : proportion d'exemples adversariaux à ajouter
    """
    model.eval()
    n_adv = int(len(X) * adv_ratio)
    indices = np.random.choice(len(X), n_adv, replace=False)

    X_adv_list = []
    for i in indices:
        x_t = torch.tensor(X[i:i+1])
        y_t = torch.tensor([y[i]])
        x_adv = fgsm_attack(model, x_t, y_t, epsilon=epsilon)
        X_adv_list.append(x_adv.numpy())

    X_adv = np.concatenate(X_adv_list, axis=0)
    y_adv = y[indices]  # labels VRAIS (l'attaque reste une attaque)

    X_aug = np.concatenate([X, X_adv], axis=0)
    y_aug = np.concatenate([y, y_adv], axis=0)
    return X_aug, y_aug


print("Génération des exemples adversariaux...")
X_robust, y_robust = generate_adversarial_dataset(model, X_train, y_train, epsilon=0.05)
print(f"Dataset augmenté : {len(X_robust)} séquences (+ {len(X_robust)-len(X_train)} adversariales)")

print("\nEntraînement du modèle robuste...")
model_robust = LSTM_IDS()
train_model(model_robust, X_robust, y_robust, epochs=20)

In [ ]:
# ── Comparaison finale : modèle standard vs modèle robuste ───────────────────

def evaluate_robustness(model, X_test, y_test, epsilons=[0.0, 0.01, 0.05, 0.10, 0.20]):
    """Évalue la précision du modèle sous attaque FGSM de différentes intensités."""
    model.eval()
    results = {}
    attack_idx = np.where(y_test == 1)[0][:100]

    for eps in epsilons:
        correct = 0
        for i in attack_idx:
            x = torch.tensor(X_test[i:i+1])
            y = torch.tensor([y_test[i]])
            if eps == 0.0:
                x_eval = x
            else:
                x_eval = fgsm_attack(model, x, y, epsilon=eps)
            with torch.no_grad():
                pred = model(x_eval).item()
            if pred > 0.5:
                correct += 1
        results[eps] = correct / len(attack_idx) * 100
    return results

print("Évaluation en cours...")
res_std    = evaluate_robustness(model, X_test, y_test)
res_robust = evaluate_robustness(model_robust, X_test, y_test)

# Visualisation
epsilons = list(res_std.keys())
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epsilons, list(res_std.values()),    'o-', color='tomato',    label='Modèle standard', linewidth=2)
ax.plot(epsilons, list(res_robust.values()), 's-', color='steelblue', label='Modèle robuste (adv. training)', linewidth=2)
ax.axhline(50, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.set_xlabel('Amplitude de perturbation ε', fontsize=11)
ax.set_ylabel('Taux de détection des attaques (%)', fontsize=11)
ax.set_title('Robustesse sous attaque FGSM : standard vs adversarial training', fontsize=12)
ax.legend()
ax.set_ylim(0, 105)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('robustness_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n📊 Taux de détection sous FGSM :")
print(f"{'ε':>6} | {'Standard':>12} | {'Robuste':>12}")
print("-" * 36)
for eps in epsilons:
    print(f"{eps:>6.2f} | {res_std[eps]:>11.1f}% | {res_robust[eps]:>11.1f}%")

### ✏️ Question 3

1. À ε=0.05, quelle est la différence de robustesse entre les deux modèles ?
2. L'adversarial training a-t-il un coût ? Comparez la précision sur les exemples **non perturbés** (ε=0.0).
3. Pourquoi un attaquant qui connaît que vous utilisez l'adversarial training peut-il contourner cette défense ?

```python
# Votre réponse ici
# 1. ???
# 2. ???
# 3. ???
```

---
## Partie 5 — Bilan MITRE ATLAS

### ✏️ Exercice final : remplir la matrice

Vous avez réalisé deux types d'attaques. Complétez la fiche de rapport ci-dessous,  
comme le ferait un red team documentant ses travaux pour le RSSI.

```
═══════════════════════════════════════════════════════════════
RAPPORT RED TEAM — LSTM-IDS ACARS
═══════════════════════════════════════════════════════════════

ATTAQUE 1 — Evasion (FGSM)
───────────────────────────────────────────────────────────────
Tactique ATLAS      : Impact
Technique           : AML.T0015 — Evade ML Model
Phase               : Production (white-box)
Pré-requis attaquant: ???
Impact opérationnel : ???
Détectabilité       : ???
Contre-mesure       : ???
Résidu de risque    : ???

ATTAQUE 2 — Backdoor (Data Poisoning)
───────────────────────────────────────────────────────────────
Tactique ATLAS      : Persistence
Technique           : AML.T0020 — Poison Training Data
Phase               : Pipeline d'entraînement (supply-chain)
Pré-requis attaquant: ???
Impact opérationnel : ???
Détectabilité       : ???
Contre-mesure       : ???
Résidu de risque    : ???

RECOMMANDATIONS (par ordre de priorité)
───────────────────────────────────────────────────────────────
1. ???
2. ???
3. ???
═══════════════════════════════════════════════════════════════
```

---
## Pour aller plus loin

- **PGD** (Projected Gradient Descent) : version itérée de FGSM, bien plus puissante
- **Randomized Smoothing** : défense certifiable mathématiquement (Cohen et al. 2019)
- **MagNet** : détection d'inputs adversariaux par auto-encodeur (ironie : un AE pour défendre un LSTM !)
- **MITRE ATLAS** : `https://atlas.mitre.org` — explorer les cas réels documentés
- **Adversarial Robustness Toolbox (IBM ART)** : bibliothèque Python complète pour tester et défendre

```python
# pip install adversarial-robustness-toolbox
# from art.attacks.evasion import FastGradientMethod
# from art.defences.trainer import AdversarialTrainer
```

---
*Module 3 — Deep Learning · M1 IA & Cybersécurité Aéronautique & Spatial*